In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.model_selection import train_test_split

torch.manual_seed(42)

print("PyTorch:", torch.__version__)
print("Підготовка розпочата.")

In [ ]:
# Завантаження Chest X-Ray Pneumonia з Kaggle

try:
    import kagglehub

    dataset_path = Path(
        kagglehub.dataset_download(
            "paultimothymooney/chest-xray-pneumonia"
        )
    )
    print("Датасет завантажено:")
    print(dataset_path)

except Exception as e:
    print("Автоматичне завантаження не виконалося.")
    print("Помилка:", e)
    print()
    print("Якщо датасет завантажений вручну, вкажи шлях нижче.")

    DATASET_PATH = Path("./chest_xray")
    dataset_path = DATASET_PATH

    if not dataset_path.exists():
        raise FileNotFoundError(
            "Не знайдено датасет. Вкажи правильний шлях у DATASET_PATH."
        )

In [ ]:
# Збираємо шляхи до всіх зображень
# 0 = NORMAL
# 1 = PNEUMONIA

image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}

image_paths = []
labels = []

for split_name in ["train", "test"]:
    split_dir = dataset_path / "chest_xray" / split_name

    if not split_dir.exists():
        split_dir = dataset_path / split_name

    if not split_dir.exists():
        continue

    for class_name, label in [("NORMAL", 0), ("PNEUMONIA", 1)]:
        class_dir = split_dir / class_name

        if class_dir.exists():
            for path in class_dir.rglob("*"):
                if path.suffix.lower() in image_extensions:
                    image_paths.append(path)
                    labels.append(label)

print("Всього зображень:", len(image_paths))
print("NORMAL:", labels.count(0))
print("PNEUMONIA:", labels.count(1))

if len(image_paths) == 0:
    raise RuntimeError("Зображення не знайдені. Перевір шлях до датасету.")

In [ ]:
# Перевірка початкового зображення

with Image.open(image_paths[0]) as img:
    print("Файл:", image_paths[0])
    print("Початковий розмір:", img.size)
    print("Режим:", img.mode)

    plt.figure(figsize=(5, 5))
    plt.imshow(img, cmap="gray")
    plt.title("Приклад рентген-знімка")
    plt.axis("off")
    plt.show()

In [ ]:
train_paths, test_paths, train_labels, test_labels = train_test_split(
    image_paths,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Train:", len(train_paths))
print("Test:", len(test_paths))

print("\nTrain:")
print("NORMAL:", train_labels.count(0))
print("PNEUMONIA:", train_labels.count(1))

print("\nTest:")
print("NORMAL:", test_labels.count(0))
print("PNEUMONIA:", test_labels.count(1))

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transform створено.")

In [ ]:
class ChestXRayDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = list(paths)
        self.labels = list(labels)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = Image.open(self.paths[index]).convert("RGB")
        label = self.labels[index]

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_dataset = ChestXRayDataset(
    train_paths, train_labels, train_transform
)

test_dataset = ChestXRayDataset(
    test_paths, test_labels, test_transform
)

print("Train dataset:", len(train_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
# DataLoader для роботи з батчами

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

images, labels_batch = next(iter(train_loader))

print("Batch size:", batch_size)
print("Форма batch зображень:", images.shape)
print("Форма batch labels:", labels_batch.shape)

In [ ]:
# Фінальна перевірка

print("==========================================")
print("РЕЗУЛЬТАТ ПІДГОТОВКИ ДАТАСЕТУ")
print("==========================================")
print(f"Train: {len(train_dataset)} зображень")
print(f"Test:  {len(test_dataset)} зображень")
print(f"Розмір: {images.shape[2]}x{images.shape[3]}")
print(f"Канали: {images.shape[1]}")
print(f"Batch size: {batch_size}")
print("Розділення: 80/20")
print("Random state: 42")
print("Нормалізація: ImageNet mean/std")
print("==========================================")

In [ ]:
# Візуалізація підготовлених зображень

mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

shown = torch.clamp(images[:6].cpu() * std + mean, 0, 1)

plt.figure(figsize=(12, 6))

for i in range(min(6, len(shown))):
    plt.subplot(2, 3, i + 1)
    plt.imshow(shown[i].permute(1, 2, 0))
    title = "PNEUMONIA" if labels_batch[i].item() == 1 else "NORMAL"
    plt.title(title)
    plt.axis("off")

plt.tight_layout()
plt.show()